# ReLaFIS classification notebook

Train and evaluate **ReLaFIS** (relational antecedents + TSK consequent) using only
source modules present in this repository branch.


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import os
import sys
from pathlib import Path

# Locate repository root: nearest parent that contains model/ReLaFIS.py
_notebook_dir = Path.cwd().resolve()
_candidates = [_notebook_dir, *_notebook_dir.parents]
_root = None
for candidate in _candidates:
    if (candidate / "model" / "ReLaFIS.py").is_file():
        _root = candidate
        break
if _root is None:
    raise FileNotFoundError(
        "Could not locate repository root containing model/ReLaFIS.py"
    )

os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

print(f"Repository root: {_root}")


In [ ]:
import math
import random
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    roc_auc_score,
)
from torch.utils.data import DataLoader

from model.ReLaFIS import ReLaFIS, SklearnReLaFISWrapper
from train.early_stop import EarlyStopping
from tests.tabular_small import Iris, PimaDiabetes


In [ ]:
# ---------------------------------------------------------------------------
# Experiment configuration
# ---------------------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 32
random_state = 199
learning_rate = 0.005
epochs = 100
n_rules = 2
lambda_rec = 1.0
patience = 10
debug_checks = True   # relational-state assertions during smoke / debug
debug_gradients = False
SMOKE_TEST = True     # short Iris run for clean-clone verification
RUN_FULL_EXPERIMENT = False  # guard the 30-run evaluation cell

print(f"device={device}")


def create_experiment(seed: int):
    """Factory for the active dataset.

    Pima Case 1 uses the intended 50% / 25% / 25% stratified protocol.
    Substitute another existing dataset class here if needed, e.g. Iris(...).
    """
    return PimaDiabetes(
        session_id=seed,
        split_ratios=(0.5, 0.25, 0.25),
    )


def create_smoke_experiment(seed: int):
    """Small multiclass dataset for notebook smoke tests."""
    return Iris(
        session_id=seed,
        split_ratios=(0.5, 0.25, 0.25),
    )


In [ ]:
def set_random_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _encode_labels_like_train(experiment, y_raw):
    """Map raw validation labels onto the train/test integer encoding.

    ``val_numpy`` returns untransformed targets when a three-way split is
    configured, while train/test use PyCaret-transformed integers.
    """
    y_raw = np.asarray(y_raw)
    if np.issubdtype(y_raw.dtype, np.number):
        return y_raw.astype(np.float32)

    y_train_raw = np.asarray(experiment.clf.y_train)
    y_train_enc = np.asarray(experiment.clf.y_train_transformed).reshape(-1)
    mapping = {}
    for raw, enc in zip(y_train_raw.reshape(-1), y_train_enc):
        mapping[raw] = enc
        mapping[str(raw)] = enc
    encoded = np.array([mapping[v] if v in mapping else mapping[str(v)] for v in y_raw])
    return encoded.astype(np.float32)


def _tensor_dataset(X, y, device):
    from torch.utils.data import TensorDataset

    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)
    return TensorDataset(
        torch.tensor(X, dtype=torch.float32, device=device),
        torch.tensor(y, dtype=torch.float32, device=device),
    )


def build_dataloaders(experiment, seed: int, batch_size: int, device):
    """Build train / validation / test loaders.

    Prefer the experiment's built-in validation split when available.
    Otherwise carve a stratified validation set from training data only.
    """
    if hasattr(experiment, "has_validation_split") and experiment.has_validation_split():
        X_train, y_train = experiment.train_numpy()
        X_val, y_val_raw = experiment.val_numpy()
        X_test, y_test = experiment.test_numpy()
        y_val = _encode_labels_like_train(experiment, y_val_raw)
        train_dataset = _tensor_dataset(X_train, y_train, device)
        validation_dataset = _tensor_dataset(X_val, y_val, device)
        test_dataset = _tensor_dataset(X_test, y_test, device)
    else:
        from sklearn.model_selection import train_test_split

        X_train, y_train = experiment.train_numpy()
        X_test, y_test = experiment.test_numpy()
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train,
            y_train,
            test_size=0.25,
            stratify=y_train,
            random_state=seed,
        )
        train_dataset = _tensor_dataset(X_tr, y_tr, device)
        validation_dataset = _tensor_dataset(X_val, y_val, device)
        test_dataset = _tensor_dataset(X_test, y_test, device)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    validation_loader = DataLoader(
        validation_dataset, batch_size=batch_size, shuffle=False
    )
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    return train_loader, validation_loader, test_loader


def _is_binary_labels(y: np.ndarray) -> bool:
    return len(np.unique(y)) == 2


def _safe_auc(y_true, y_score, binary: bool) -> float:
    try:
        if binary:
            return float(roc_auc_score(y_true, y_score))
        return float(roc_auc_score(y_true, y_score, multi_class="ovr", average="macro"))
    except ValueError:
        return float("nan")


def _rule_firing_entropy(phi: torch.Tensor) -> float:
    """Normalized rule-firing entropy (collapse diagnostic only; not ARE)."""
    eps = torch.finfo(phi.dtype).eps
    safe = phi.clamp_min(eps)
    H = -(safe * torch.log(safe)).sum(dim=1)
    # normalize by log(R) when R > 1
    R = phi.size(1)
    if R > 1:
        H = H / math.log(R)
    return float(H.mean().item())


def classification_and_reconstruction_loss(
    model,
    batch_X,
    batch_y,
    binary: bool,
    binary_criterion,
    multiclass_criterion,
    reconstruction_criterion,
    lambda_rec: float,
):
    logits, reconstructed_X = model(batch_X)

    if binary:
        binary_logits = logits.squeeze(-1)
        binary_targets = batch_y.float().reshape_as(binary_logits)
        classification_loss = binary_criterion(binary_logits, binary_targets)
    else:
        classification_loss = multiclass_criterion(
            logits, batch_y.long().view(-1)
        )

    reconstruction_loss = reconstruction_criterion(reconstructed_X, batch_X)
    total_loss = classification_loss + lambda_rec * reconstruction_loss
    return total_loss, classification_loss, reconstruction_loss, logits


@torch.no_grad()
def evaluate_loader_loss(model, loader, binary, binary_criterion, multiclass_criterion):
    model.eval()
    total = 0.0
    n = 0
    for batch_X, batch_y in loader:
        logits, _ = model(batch_X)
        if binary:
            binary_logits = logits.squeeze(-1)
            binary_targets = batch_y.float().reshape_as(binary_logits)
            loss = binary_criterion(binary_logits, binary_targets)
        else:
            loss = multiclass_criterion(logits, batch_y.long().view(-1))
        total += loss.item() * batch_X.size(0)
        n += batch_X.size(0)
    return total / max(n, 1)


def evaluate_model(model, experiment, device, binary: bool):
    """Final train/test metrics after model selection."""
    model.eval()
    wrapper = SklearnReLaFISWrapper(model, device=device, dtype=torch.float32)

    X_train, y_train = experiment.train_numpy()
    X_test, y_test = experiment.test_numpy()

    train_pred = wrapper.predict(X_train)
    test_pred = wrapper.predict(X_test)
    train_proba = wrapper.predict_proba(X_train)
    test_proba = wrapper.predict_proba(X_test)

    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)

    if binary:
        train_auc = _safe_auc(y_train, train_proba[:, 1], binary=True)
        test_auc = _safe_auc(y_test, test_proba[:, 1], binary=True)
    else:
        train_auc = _safe_auc(y_train, train_proba, binary=False)
        test_auc = _safe_auc(y_test, test_proba, binary=False)

    params = model.get_interpretable_params()
    are = model.linguistic_richness()
    relax = model.relaxation_rate()

    return {
        "train_accuracy": train_acc,
        "test_accuracy": test_acc,
        "train_auc": train_auc,
        "test_auc": test_auc,
        "train_report": classification_report(y_train, train_pred, zero_division=0),
        "test_report": classification_report(y_test, test_pred, zero_division=0),
        "are": are,
        "relaxation_rate": relax,
        "rho_mean": params["rho_mean"],
        "rho_std": params["rho_std"],
        "sigma_mean": params["sigma_mean"],
        "sigma_std": params["sigma_std"],
        "params": params,
    }


def train_one_model(
    *,
    experiment,
    seed: int,
    device,
    batch_size: int,
    n_rules: int,
    learning_rate: float,
    epochs: int,
    lambda_rec: float,
    patience: int,
    debug_checks: bool = False,
    debug_gradients: bool = False,
    max_epochs_override: int | None = None,
):
    set_random_seed(seed)
    train_loader, validation_loader, test_loader = build_dataloaders(
        experiment, seed, batch_size, device
    )

    X_train, y_train = experiment.train_numpy()
    n_features = X_train.shape[1]
    n_classes = len(np.unique(y_train))
    binary = _is_binary_labels(y_train)

    model = ReLaFIS(
        in_features=n_features,
        rules=n_rules,
        out_features=n_classes,
        binary=binary,
        dtype=torch.float32,
        device=device,
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    binary_criterion = nn.BCEWithLogitsLoss()
    multiclass_criterion = nn.CrossEntropyLoss()
    reconstruction_criterion = nn.MSELoss()
    early_stopping = EarlyStopping(patience=patience, delta=0.0)

    history = {
        "train_cls": [],
        "train_rec": [],
        "train_total": [],
        "val_cls": [],
        "are": [],
        "relaxation_rate": [],
        "rho_mean": [],
        "rho_std": [],
        "sigma_mean": [],
        "sigma_std": [],
        "max_phi": [],
        "rule_firing_entropy": [],
    }

    n_epochs = max_epochs_override if max_epochs_override is not None else epochs

    for epoch in range(n_epochs):
        model.train()
        sum_cls = 0.0
        sum_rec = 0.0
        sum_total = 0.0
        n_seen = 0
        max_phi = 0.0
        firing_H_sum = 0.0

        for batch_idx, (batch_X, batch_y) in enumerate(train_loader):
            optimizer.zero_grad()
            total_loss, cls_loss, rec_loss, _ = classification_and_reconstruction_loss(
                model,
                batch_X,
                batch_y,
                binary,
                binary_criterion,
                multiclass_criterion,
                reconstruction_criterion,
                lambda_rec,
            )

            if not torch.isfinite(total_loss):
                raise FloatingPointError(
                    f"Non-finite total_loss at epoch={epoch}, batch={batch_idx}"
                )

            total_loss.backward()

            if debug_gradients:
                norms = []
                for name, param in model.named_parameters():
                    if param.grad is not None:
                        norms.append((name, float(param.grad.norm().item())))
                print(f"[grad] epoch={epoch} batch={batch_idx} {norms[:5]}...")

            optimizer.step()

            bs = batch_X.size(0)
            sum_cls += cls_loss.item() * bs
            sum_rec += rec_loss.item() * bs
            sum_total += total_loss.item() * bs
            n_seen += bs

            if debug_checks and batch_idx == 0 and epoch % 10 == 0:
                state = model._compute_relational_state(batch_X)
                assert torch.isfinite(state["phi"]).all()
                assert torch.isfinite(state["rho"]).all()
                assert torch.isfinite(state["sigma"]).all()
                assert torch.allclose(
                    state["phi"].sum(dim=1),
                    torch.ones(batch_X.size(0), device=batch_X.device),
                    atol=1e-5,
                )

            with torch.no_grad():
                state = model._compute_relational_state(batch_X)
                max_phi = max(max_phi, float(state["phi"].max().item()))
                firing_H_sum += _rule_firing_entropy(state["phi"]) * bs

        train_cls = sum_cls / max(n_seen, 1)
        train_rec = sum_rec / max(n_seen, 1)
        train_total = sum_total / max(n_seen, 1)
        val_cls = evaluate_loader_loss(
            model, validation_loader, binary, binary_criterion, multiclass_criterion
        )

        params = model.get_interpretable_params()
        history["train_cls"].append(train_cls)
        history["train_rec"].append(train_rec)
        history["train_total"].append(train_total)
        history["val_cls"].append(val_cls)
        history["are"].append(params["linguistic_richness"])
        history["relaxation_rate"].append(params["relaxation_rate"])
        history["rho_mean"].append(params["rho_mean"])
        history["rho_std"].append(params["rho_std"])
        history["sigma_mean"].append(params["sigma_mean"])
        history["sigma_std"].append(params["sigma_std"])
        history["max_phi"].append(max_phi)
        history["rule_firing_entropy"].append(firing_H_sum / max(n_seen, 1))

        early_stopping(val_cls, model)
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(
                f"epoch {epoch+1:03d} | "
                f"train_cls={train_cls:.4f} train_mse={train_rec:.4f} "
                f"total={train_total:.4f} | val_cls={val_cls:.4f} | "
                f"ARE={params['linguistic_richness']:.4f} "
                f"relax={params['relaxation_rate']:.4f} "
                f"rho={params['rho_mean']:.4f}±{params['rho_std']:.4f}"
            )

        if early_stopping.early_stop:
            print(f"Early stopping at epoch {epoch+1}")
            break

    early_stopping.load_best_model(model)
    metrics = evaluate_model(model, experiment, device, binary)
    metrics["epochs_trained"] = len(history["train_cls"])
    metrics["history"] = history
    metrics["binary"] = binary
    metrics["model"] = model
    metrics["test_loader"] = test_loader
    return metrics


## Single-run training

Uses PimaDiabetes with the official 50/25/25 protocol, unless `SMOKE_TEST` is
enabled (Iris, few epochs).


In [ ]:
if SMOKE_TEST:
    experiment = create_smoke_experiment(random_state)
    smoke_epochs = 2
else:
    experiment = create_experiment(random_state)
    smoke_epochs = None

print(experiment.split_description())
print(
    "train/val/test sizes:",
    len(experiment.train_numpy()[0]),
    len(experiment.val_numpy()[0]) if experiment.has_validation_split() else "n/a",
    len(experiment.test_numpy()[0]),
)

y_train = experiment.train_numpy()[1]
binary = _is_binary_labels(y_train)
print(f"binary={binary}, n_classes={len(np.unique(y_train))}")

single_run = train_one_model(
    experiment=experiment,
    seed=random_state,
    device=device,
    batch_size=batch_size,
    n_rules=n_rules,
    learning_rate=learning_rate,
    epochs=epochs,
    lambda_rec=lambda_rec,
    patience=patience,
    debug_checks=debug_checks,
    debug_gradients=debug_gradients,
    max_epochs_override=smoke_epochs,
)

model = single_run["model"]
print("\nTrain classification report:\n", single_run["train_report"])
print("\nTest classification report:\n", single_run["test_report"])
print(
    f"test_acc={single_run['test_accuracy']:.4f} "
    f"test_auc={single_run['test_auc']:.4f} "
    f"ARE={single_run['are']:.4f} "
    f"relax={single_run['relaxation_rate']:.4f}"
)


In [ ]:
# Diagnostic plots (matplotlib only; no missing utils modules)
history = single_run["history"]
epochs_axis = np.arange(1, len(history["train_cls"]) + 1)

fig, axes = plt.subplots(2, 2, figsize=(10, 7))

axes[0, 0].plot(epochs_axis, history["train_cls"], label="train cls")
axes[0, 0].plot(epochs_axis, history["val_cls"], label="val cls")
axes[0, 0].set_title("Classification loss")
axes[0, 0].legend()

axes[0, 1].plot(epochs_axis, history["train_rec"], label="train MSE")
axes[0, 1].plot(epochs_axis, history["train_total"], label="total")
axes[0, 1].set_title("Reconstruction / total")
axes[0, 1].legend()

axes[1, 0].plot(epochs_axis, history["are"], label="ARE")
axes[1, 0].plot(epochs_axis, history["relaxation_rate"], label="relaxation")
axes[1, 0].set_title("ARE & relaxation rate")
axes[1, 0].legend()

axes[1, 1].plot(epochs_axis, history["rho_mean"], label="rho mean")
axes[1, 1].fill_between(
    epochs_axis,
    np.array(history["rho_mean"]) - np.array(history["rho_std"]),
    np.array(history["rho_mean"]) + np.array(history["rho_std"]),
    alpha=0.2,
)
axes[1, 1].plot(
    epochs_axis,
    history["rule_firing_entropy"],
    label="rule-firing entropy (collapse diag.)",
)
axes[1, 1].set_title("rho & rule-firing entropy")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print("Interpretable parameters:")
for key in (
    "linguistic_richness",
    "relaxation_rate",
    "rho_mean",
    "rho_std",
    "sigma_mean",
    "sigma_std",
    "w1_mean",
    "w2_mean",
    "w3_mean",
):
    print(f"  {key}: {single_run['params'][key]}")


In [ ]:
# Persist single-run summary
Path("results").mkdir(parents=True, exist_ok=True)
result_path = Path("results") / f"relafis_{experiment.__class__.__name__}_{n_rules}.txt"
result = (
    f"dataset={experiment.__class__.__name__}\n"
    f"split={experiment.split_description()}\n"
    f"rules={n_rules}\n"
    f"epochs_trained={single_run['epochs_trained']}\n"
    f"test_accuracy={single_run['test_accuracy']:.6f}\n"
    f"test_auc={single_run['test_auc']:.6f}\n"
    f"ARE={single_run['are']:.6f}\n"
    f"relaxation_rate={single_run['relaxation_rate']:.6f}\n"
    f"rho_mean={single_run['rho_mean']:.6f}\n"
    f"sigma_mean={single_run['sigma_mean']:.6f}\n"
    f"\nTrain report\n{single_run['train_report']}\n"
    f"\nTest report\n{single_run['test_report']}\n"
)
result_path.write_text(result)
print(f"Wrote {result_path}")


## Ordinary 30-run evaluation

Guarded by `RUN_FULL_EXPERIMENT`. Uses distinct reproducible seeds and reports
`mean ± population std` over all runs (no top-k selection).


In [ ]:
BASE_SEED = 199
N_RUNS = 30
run_seeds = [BASE_SEED + index for index in range(N_RUNS)]


def run_repeated_experiment(seeds, **train_kwargs):
    records = []
    for index, seed in enumerate(seeds):
        print(f"\n=== Run {index+1}/{len(seeds)} seed={seed} ===")
        experiment = create_experiment(seed)
        metrics = train_one_model(
            experiment=experiment,
            seed=seed,
            **train_kwargs,
        )
        records.append(
            {
                "seed": seed,
                "test_accuracy": metrics["test_accuracy"],
                "test_auc": metrics["test_auc"],
                "are": metrics["are"],
                "relaxation_rate": metrics["relaxation_rate"],
                "rho_mean": metrics["rho_mean"],
                "sigma_mean": metrics["sigma_mean"],
                "epochs_trained": metrics["epochs_trained"],
            }
        )
        # Explicitly drop model to avoid accidental reuse
        del metrics["model"]
    return records


def summarize_runs(records):
    keys = [
        "test_accuracy",
        "test_auc",
        "are",
        "relaxation_rate",
        "rho_mean",
        "sigma_mean",
        "epochs_trained",
    ]
    summary = {}
    for key in keys:
        values = np.array([r[key] for r in records], dtype=float)
        mean = float(np.mean(values))
        std = float(np.std(values, ddof=0))
        summary[key] = (mean, std)
        print(f"{key}: {mean:.6f} ± {std:.6f}")
    return summary


if RUN_FULL_EXPERIMENT:
    results_30_runs = run_repeated_experiment(
        run_seeds,
        device=device,
        batch_size=batch_size,
        n_rules=n_rules,
        learning_rate=learning_rate,
        epochs=epochs,
        lambda_rec=lambda_rec,
        patience=patience,
        debug_checks=False,
        debug_gradients=False,
    )
    Path("results").mkdir(parents=True, exist_ok=True)
    summary = summarize_runs(results_30_runs)
    out = Path("results") / "relafis_30runs_summary.txt"
    lines = ["mean ± population standard deviation", ""]
    for key, (mean, std) in summary.items():
        lines.append(f"{key}: {mean:.6f} ± {std:.6f}")
    out.write_text("\n".join(lines) + "\n")
    print(f"Wrote {out}")
else:
    print(
        "RUN_FULL_EXPERIMENT is False — skipping the 30-run evaluation. "
        "Set RUN_FULL_EXPERIMENT = True to execute all ordinary runs."
    )
